# Community Structure Sweeps - CIC3 Metrics (degree-corrected)

Sweep community structure while **holding average degree approximately
fixed**. Both `p_intra` and `p_inter` move along a 1D curve that keeps
the expected total edge count constant: as `p_inter` rises, `p_intra`
falls to compensate.

## Network design

- **$K=25$ communities of size $n=80$** (fewer, larger communities than
  the original 50x40 setup). Resulting $k_{\text{avg}} \approx 79$ when
  intra-cliques are full, giving a meaningfully denser network.
- **$p_{\text{tri}} = 0.003$** (down from 0.05) so triangles fill only
  ~21% of intra-edge slots (down from ~86%) - triangles no longer dominate
  the density budget, which gives a wider feasible `p_inter` range.
- **$T_{\text{max}} = 300$** so simulations cut off in a reasonable time
  when isolated communities trap part of the network and quotas can't be
  met.

## Why a constant-edge-budget sweep

The original sweep raised `p_inter` at fixed `p_intra`, which inflated
$k_{\text{avg}}$ from ~19 to ~307 across the sweep. Any structure-vs-
attainment signal was tangled with a density signal. We now move along a
curve where total expected edges (and thus $k_{\text{avg}}$) stay
constant.

Two effects need balancing:

1. **Direct edge sampling.** Variable-part pair-slots:
   - $A = K \cdot \binom{n}{2} = 79{,}000$ intra-community
   - $B = \binom{K}{2} \cdot n^{2} = 1{,}920{,}000$ inter-community

2. **Triangle fill (RSC convention).** Each intra pair has $n-2$
   candidate third nodes. A missing intra pair gets filled by *some*
   triangle with probability
   $f_{\text{tri}} = 1 - (1 - p_{\text{tri}})^{n-2} \approx 0.209$,
   regardless of `p_intra`. Effective intra-edge fraction:
   $f(p_{\text{intra}}) = p_{\text{intra}} + (1 - p_{\text{intra}}) f_{\text{tri}}$.

Constant-budget equation $A \cdot f(p_{\text{intra}}) + B \cdot p_{\text{inter}} = A$ gives:

$$p_{\text{intra}} = 1 - \frac{B/A}{1 - f_{\text{tri}}} \cdot p_{\text{inter}} \approx 1 - 30.7 \cdot p_{\text{inter}}$$

`p_inter` sweeps from 0 (well-separated full cliques, isolated communities)
up to ~0.03 (`p_intra ~ 0.08`, communities barely distinguishable).

## Plots (each averaged over runs)
1. **Attainment** vs `p_inter` - $A_g$ (faint) and $A_g^{td}$ (bold) on one axes.
2. **Deadweight** vs `p_inter` - $D_g$, trial-mean $\pm 1$ std.
3. **Penetration** vs `p_inter` - $P_g$, trial-mean $\pm 1$ std.
4. **Sanity check** - realized $k_{\text{avg}}$ across the sweep.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import pickle
import hashlib
import math
import os

from scm import SBMGenerator, MultiRandomSeeding
from scm.cic3_simulator import CIC3Simulator
from scm.analysis import (
    attainment,
    time_discounted_attainment,
    exponential_decay,
    deadweight,
    penetration,
)

# --- Cache helpers ---
CACHE_DIR = '../results/community_structure_sweeps_corrected'

def _param_fingerprint(**kwargs):
    raw = str(sorted(kwargs.items()))
    return hashlib.md5(raw.encode()).hexdigest()[:12]

def _cache_path(name):
    return os.path.join(CACHE_DIR, f'{name}.pkl')

def save_cache(name, obj, **fingerprint_kwargs):
    os.makedirs(CACHE_DIR, exist_ok=True)
    fp = _param_fingerprint(**fingerprint_kwargs)
    path = _cache_path(name)
    with open(path, 'wb') as f:
        pickle.dump({'fingerprint': fp, 'data': obj}, f)
    print(f'  Cached {name} -> {path} (fp={fp})')

def load_cache(name, **fingerprint_kwargs):
    path = _cache_path(name)
    if not os.path.exists(path):
        return None
    fp = _param_fingerprint(**fingerprint_kwargs)
    with open(path, 'rb') as f:
        blob = pickle.load(f)
    if blob['fingerprint'] != fp:
        print(f'  Cache stale for {name} (disk={blob["fingerprint"]}, current={fp})')
        return None
    print(f'  Loaded {name} from cache (fp={fp})')
    return blob['data']

## Parameters

In [ ]:
# --- Network ---
N = 2000
K_COMM = 25                       # fewer communities (was 50)
COMMUNITY_SIZE = N // K_COMM      # 80 (was 40)
COMMUNITY_SIZES = [COMMUNITY_SIZE] * K_COMM

# Triangle probability (intra only, uniform across communities, FIXED).
# Reduced from 0.05 so triangles no longer dominate intra-edge fill.
P_TRI_INTRA = 0.003

# --- Constant-edge-budget curve (with triangle-fill correction) ---
A_PAIRS = K_COMM * math.comb(COMMUNITY_SIZE, 2)
B_PAIRS = math.comb(K_COMM, 2) * COMMUNITY_SIZE ** 2
F_TRI   = 1.0 - (1.0 - P_TRI_INTRA) ** (COMMUNITY_SIZE - 2)
SLOPE   = (B_PAIRS / A_PAIRS) / (1.0 - F_TRI)

def p_intra_for(p_inter):
    return 1.0 - SLOPE * p_inter

# --- Sweep ---
# p_inter from 0 (full cliques) up to just shy of 1/SLOPE, where p_intra
# would hit 0. Stop at 0.03 so p_intra stays positive (~0.08).
P_INTER_VALS = np.linspace(0.0, 0.03, 11)
P_INTRA_VALS = p_intra_for(P_INTER_VALS)

# --- CIC3 parameters ---
C = 10
QUOTAS = [N // C] * C
SEEDS_PER_CONTAGION = 1
# Cap simulation length so isolated-community runs (where some quotas can
# never be met because parts of the network are unreachable) terminate.
T_MAX = 300
DECAY_RATE = 0.05
V = exponential_decay(DECAY_RATE)

# --- Infection (rescaled infectivity) ---
LAMBDA = 1.0
LAMBDA_DELTA = 2.0

# --- Simulation ---
NUM_TRIALS = 15
TOPO_SEED = 2025

print(f'K={K_COMM}  n={COMMUNITY_SIZE}  p_tri={P_TRI_INTRA}')
print(f'A = {A_PAIRS}  B = {B_PAIRS}')
print(f'f_tri = {F_TRI:.4f}  slope = {SLOPE:.2f}  '
      f'p_inter max (where p_intra=0) = {1/SLOPE:.5f}')
print(f't_max = {T_MAX}')
print('\nConstant-budget sweep points:')
print(f'{"p_inter":>10s}{"p_intra":>10s}')
for pi, pa in zip(P_INTER_VALS, P_INTRA_VALS):
    print(f'{pi:>10.5f}{pa:>10.4f}')

## Topology Generation

For each `p_inter` on the budget curve, generate one SBM at the paired
`p_intra`. Triangle probability is fixed.

In [ ]:
topologies = {}

for p_inter in P_INTER_VALS:
    p_intra = p_intra_for(p_inter)
    block_matrix = np.full((K_COMM, K_COMM), p_inter)
    np.fill_diagonal(block_matrix, p_intra)

    gen = SBMGenerator(
        community_sizes=COMMUNITY_SIZES,
        block_matrix=block_matrix,
        triangle_block_probs=[P_TRI_INTRA] * K_COMM,
    )
    links, triangles = gen.generate(seed=TOPO_SEED)
    topologies[round(p_inter, 6)] = {
        'links': links,
        'triangles': triangles,
        'N': gen.N,
        'k_avg': gen.k_avg,
        'k_d_avg': gen.k_delta_avg,
        'p_intra': p_intra,
        'p_inter': p_inter,
    }
    print(f'  p_inter={p_inter:.5f}  p_intra={p_intra:.4f}'
          f'  -> k_avg={gen.k_avg:.2f}  k_d_avg={gen.k_delta_avg:.2f}')

print(f'\nGenerated {len(topologies)} topologies')
k_avg_arr_realized = np.array([topologies[round(p, 6)]['k_avg']
                               for p in P_INTER_VALS])
print(f'Realized k_avg range: [{k_avg_arr_realized.min():.2f}, '
      f'{k_avg_arr_realized.max():.2f}]  '
      f'spread = {k_avg_arr_realized.max() - k_avg_arr_realized.min():.2f}')

## Run CIC3 Simulations

For each topology, run `NUM_TRIALS` independent CIC3 simulations with random
seeding. Record $A_g$, $A_g^{td}$, $D_g$, and $P_g$ for each trial so we can
trial-average and shade $\pm 1$ std on the plots.

The simulator caps each run at `T_MAX = 300` timesteps. At low `p_inter`,
many communities are effectively isolated and contagions trapped in a
community can never reach the rest of the network - the cap ensures we
move on and record whatever attainment was reached.

In [ ]:
def run_community_sweep():
    fp_kwargs = dict(
        N=N, K=K_COMM, p_inters=list(P_INTER_VALS),
        p_intras=list(P_INTRA_VALS), p_tri=P_TRI_INTRA,
        C=C, quotas=QUOTAS, seeds_per_contagion=SEEDS_PER_CONTAGION,
        t_max=T_MAX, decay_rate=DECAY_RATE, num_trials=NUM_TRIALS,
        topo_seed=TOPO_SEED, lam=LAMBDA, lam_delta=LAMBDA_DELTA,
    )
    cached = load_cache('community_sweep', **fp_kwargs)
    if cached is not None:
        return cached

    results = {}

    for i, p_inter in enumerate(P_INTER_VALS):
        print(f'    [{i + 1}/{len(P_INTER_VALS)}] '
              f'p_inter={p_inter:.5f}  p_intra={P_INTRA_VALS[i]:.4f}')

        topo = topologies[round(p_inter, 6)]
        beta = LAMBDA / topo['k_avg']
        beta_delta = LAMBDA_DELTA / topo['k_d_avg']
        betas = [beta] * C
        beta_deltas = [beta_delta] * C

        num_seeds_per_contagion = [SEEDS_PER_CONTAGION] * C

        Ag_trials, Ag_td_trials = [], []
        Dg_trials, Pg_trials = [], []

        for trial in range(NUM_TRIALS):
            seeder = MultiRandomSeeding(
                N=topo['N'],
                num_seeds_per_contagion=num_seeds_per_contagion,
                links=topo['links'],
                triangles=topo['triangles'],
            )
            seeds = seeder.seed()

            sim = CIC3Simulator(
                links=topo['links'],
                triangles=topo['triangles'],
                initial_infected_per_contagion=seeds,
                betas=betas,
                beta_deltas=beta_deltas,
                quotas=QUOTAS,
                stop_on_all_quotas_met=False,
            )
            sim.run(T_MAX)

            _, Ag = attainment(sim.infected_by, QUOTAS)
            _, Ag_td = time_discounted_attainment(
                sim.infected_by, sim.infection_times, QUOTAS, V
            )
            _, Dg = deadweight(sim.infected_by, QUOTAS)
            _, Pg = penetration(topo['links'], sim.infected_by, seeds)

            Ag_trials.append(Ag)
            Ag_td_trials.append(Ag_td)
            Dg_trials.append(Dg)
            Pg_trials.append(Pg)

        results[round(p_inter, 6)] = {
            'Ag_trials':    np.array(Ag_trials),
            'Ag_td_trials': np.array(Ag_td_trials),
            'Dg_trials':    np.array(Dg_trials),
            'Pg_trials':    np.array(Pg_trials),
        }
        print(f'      Ag={np.mean(Ag_trials):.3f}'
              f'  Ag_td={np.mean(Ag_td_trials):.3f}'
              f'  Dg={np.mean(Dg_trials):.1f}'
              f'  Pg={np.mean(Pg_trials):.2f}')

    save_cache('community_sweep', results, **fp_kwargs)
    return results

all_results = run_community_sweep()
print(f'\nDone - {len(all_results)} configurations')

## Aggregate over runs

In [ ]:
keys = [round(p, 6) for p in P_INTER_VALS]

Ag_trials_2d    = np.stack([all_results[k]['Ag_trials']    for k in keys])
Ag_td_trials_2d = np.stack([all_results[k]['Ag_td_trials'] for k in keys])
Dg_trials_2d    = np.stack([all_results[k]['Dg_trials']    for k in keys])
Pg_trials_2d    = np.stack([all_results[k]['Pg_trials']    for k in keys])

Ag_mean    = Ag_trials_2d.mean(axis=1)
Ag_std     = Ag_trials_2d.std(axis=1, ddof=1)
Ag_td_mean = Ag_td_trials_2d.mean(axis=1)
Ag_td_std  = Ag_td_trials_2d.std(axis=1, ddof=1)
Dg_mean    = Dg_trials_2d.mean(axis=1)
Dg_std     = Dg_trials_2d.std(axis=1, ddof=1)
Pg_mean    = Pg_trials_2d.mean(axis=1)
Pg_std     = Pg_trials_2d.std(axis=1, ddof=1)

print(f'Aggregated {Ag_trials_2d.shape[0]} sweep points '
      f'over {Ag_trials_2d.shape[1]} trials each.')

## Helper: top-axis annotation for paired `p_intra`

Each plot has `p_inter` on the bottom axis; we annotate the matching
`p_intra` on a twin top axis so the structure trade-off is readable at a
glance.

In [ ]:
def add_pintra_top_axis(ax):
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(P_INTER_VALS)
    ax_top.set_xticklabels([f'{v:.2f}' for v in P_INTRA_VALS],
                           fontsize=8, rotation=0)
    ax_top.set_xlabel(r'Paired intra-community $p_{\mathrm{intra}}$',
                      fontsize=10)
    return ax_top

## Attainment vs community structure (averaged over runs)

Single plot, both $A_g$ and $A_g^{td}$, trial-averaged with $\pm 1$ std
shading. The orange band between the two lines is the time-discount cost.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.8))

ax.fill_between(P_INTER_VALS, Ag_td_mean, Ag_mean,
                color='orange', alpha=0.25, label='Time-discount gap')

ax.plot(P_INTER_VALS, Ag_mean, color='steelblue', lw=1.8,
        alpha=0.55, marker='o', label=r'$A_g$ (undiscounted)')
ax.fill_between(P_INTER_VALS, Ag_mean - Ag_std, Ag_mean + Ag_std,
                color='steelblue', alpha=0.10)

ax.plot(P_INTER_VALS, Ag_td_mean, color='darkorange', lw=2.5,
        marker='o', label=r'$A_g^{td}$ (time-discounted)')
ax.fill_between(P_INTER_VALS, Ag_td_mean - Ag_td_std,
                Ag_td_mean + Ag_td_std, color='darkorange', alpha=0.18)

ax.set_xlim(P_INTER_VALS[0], P_INTER_VALS[-1])
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel(r'Inter-community edge probability, $p_{\mathrm{inter}}$',
              fontsize=12)
ax.set_ylabel(r'Global attainment', fontsize=12)
ax.set_title(
    rf'CIC3 Attainment vs Community Structure '
    rf'(constant-edge-budget, avg over {NUM_TRIALS} runs)'
    '\n'
    rf'($N={N}$, $K={K_COMM}$, $C={C}$, '
    rf'$\lambda={LAMBDA}$, $\lambda_\Delta={LAMBDA_DELTA}$)',
    fontsize=11, pad=18,
)
ax.legend(loc='lower right', framealpha=1, edgecolor='black', fontsize=9)
ax.grid(alpha=0.3)
add_pintra_top_axis(ax)
plt.tight_layout()
out_path = '../figures/community_sweep_corrected_attainment.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

## Deadweight vs community structure (averaged over runs)

$D_g$ = mean number of excess infections beyond quota per contagion.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.8))

ax.plot(P_INTER_VALS, Dg_mean, color='darkred', lw=2.5,
        marker='o', label=r'$D_g$ (avg over runs)')
ax.fill_between(P_INTER_VALS, Dg_mean - Dg_std, Dg_mean + Dg_std,
                color='darkred', alpha=0.18, label=r'$\pm 1$ std')

ax.set_xlim(P_INTER_VALS[0], P_INTER_VALS[-1])
ax.set_xlabel(r'Inter-community edge probability, $p_{\mathrm{inter}}$',
              fontsize=12)
ax.set_ylabel(r'Deadweight, $D_g$', fontsize=12)
ax.set_title(
    rf'CIC3 Deadweight vs Community Structure '
    rf'(constant-edge-budget, avg over {NUM_TRIALS} runs)'
    '\n'
    rf'($N={N}$, $K={K_COMM}$, $C={C}$, '
    rf'$\lambda={LAMBDA}$, $\lambda_\Delta={LAMBDA_DELTA}$)',
    fontsize=11, pad=18,
)
ax.legend(loc='best', framealpha=1, edgecolor='black', fontsize=9)
ax.grid(alpha=0.3)
add_pintra_top_axis(ax)
plt.tight_layout()
out_path = '../figures/community_sweep_corrected_deadweight.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

## Penetration vs community structure (averaged over runs)

$P_g$ = mean BFS hop distance from seeds through each contagion's
infected subgraph.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.8))

ax.plot(P_INTER_VALS, Pg_mean, color='darkgreen', lw=2.5,
        marker='o', label=r'$P_g$ (avg over runs)')
ax.fill_between(P_INTER_VALS, Pg_mean - Pg_std, Pg_mean + Pg_std,
                color='darkgreen', alpha=0.18, label=r'$\pm 1$ std')

ax.set_xlim(P_INTER_VALS[0], P_INTER_VALS[-1])
ax.set_xlabel(r'Inter-community edge probability, $p_{\mathrm{inter}}$',
              fontsize=12)
ax.set_ylabel(r'Penetration, $P_g$', fontsize=12)
ax.set_title(
    rf'CIC3 Penetration vs Community Structure '
    rf'(constant-edge-budget, avg over {NUM_TRIALS} runs)'
    '\n'
    rf'($N={N}$, $K={K_COMM}$, $C={C}$, '
    rf'$\lambda={LAMBDA}$, $\lambda_\Delta={LAMBDA_DELTA}$)',
    fontsize=11, pad=18,
)
ax.legend(loc='best', framealpha=1, edgecolor='black', fontsize=9)
ax.grid(alpha=0.3)
add_pintra_top_axis(ax)
plt.tight_layout()
out_path = '../figures/community_sweep_corrected_penetration.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

## Sanity check: realized $k_{\text{avg}}$ across the sweep

The whole point of the constant-edge-budget design is that this should be
nearly flat. Small drift is expected from triangle/edge sampling overlap
that the analytical correction does not fully absorb.

In [ ]:
k_avg_arr = np.array([topologies[round(p, 6)]['k_avg']   for p in P_INTER_VALS])
k_d_arr   = np.array([topologies[round(p, 6)]['k_d_avg'] for p in P_INTER_VALS])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(P_INTER_VALS, k_avg_arr, marker='o', color='tab:blue', lw=2)
ax1.set_xlabel(r'$p_{\mathrm{inter}}$')
ax1.set_ylabel(r'Realized $\langle k \rangle$')
ax1.set_title(r'Realized $k_{\text{avg}}$')
ax1.grid(alpha=0.3)
ax1.axhline(k_avg_arr.mean(), color='black', lw=0.8, ls='--',
            label=f'mean = {k_avg_arr.mean():.2f}')
ax1.legend(loc='best')

ax2.plot(P_INTER_VALS, k_d_arr, marker='s', color='tab:orange', lw=2)
ax2.set_xlabel(r'$p_{\mathrm{inter}}$')
ax2.set_ylabel(r'Realized $\langle k_\Delta \rangle$')
ax2.set_title(r'Realized $k_{\Delta,\text{avg}}$')
ax2.grid(alpha=0.3)
ax2.axhline(k_d_arr.mean(), color='black', lw=0.8, ls='--',
            label=f'mean = {k_d_arr.mean():.2f}')
ax2.legend(loc='best')

plt.tight_layout()
out_path = '../figures/community_sweep_corrected_kavg_check.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

print(f'\nk_avg range: [{k_avg_arr.min():.2f}, {k_avg_arr.max():.2f}]  '
      f'spread = {k_avg_arr.max() - k_avg_arr.min():.2f}')
print(f'k_delta range: [{k_d_arr.min():.2f}, {k_d_arr.max():.2f}]  '
      f'spread = {k_d_arr.max() - k_d_arr.min():.2f}')